# ItemCF：算法、评估与推荐

先完成 `01_data_analysis.ipynb`，再在项目根目录运行 `python train.py --model itemcf`。
本 Notebook 读取训练结果，并支持修改用户 ID 查看推荐。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / 'src' / 'models' / 'item_cf.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'itemcf_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model itemcf。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新运行 python train.py --model itemcf。')

## 1. 算法与当前参数

ItemCF 使用训练期评分 ≥4 的二值矩阵计算电影共现余弦相似度，去掉自身相似度，每部来源电影保留 50 个邻居。当前收缩参数为 0。

候选得分为高分历史的相似度贡献之和，并乘以时间权重 `2 ** (-age_days / 90)`；age 从该用户最近已知评分时间计算。半衰期按验证 NDCG@10 选中 90 天。过滤所有已评分电影，测试评分仅使用当时已知训练＋验证历史。


In [2]:
search = pd.read_csv(TABLE_DIR / 'itemcf_validation_search.csv')
display(search[['half_life_days', 'n_neighbors', 'shrinkage', 'Precision@10', 'Recall@10', 'NDCG@10', 'selected']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}', 'NDCG@10': '{:.4f}'}))


,half_life_days,n_neighbors,shrinkage,Precision@10,Recall@10,NDCG@10,selected
0,0.000000,50,0,5.97%,7.89%,0.0844,False
1,1.000000,50,0,5.92%,7.96%,0.0841,False
2,7.000000,50,0,5.97%,7.95%,0.0843,False
3,30.000000,50,0,6.00%,7.94%,0.0847,False
4,90.000000,50,0,6.00%,7.91%,0.0847,True


## 2. 评估结果

在完整未评分候选目录上评估前 10 部推荐，各指标先按用户计算再取平均：

- **Precision@10**：命中数 ÷ 10。
- **Recall@10**：命中数 ÷ 该用户留出集中的高分目标数。
- **NDCG@10**：考虑命中位置，越靠前越好，并按理想排序归一化。

验证使用训练历史，测试使用训练加验证历史，模型参数仅在训练集拟合。
未评分电影不计为已知命中，但其真实偏好未知；指标不等于实际满意度。

In [3]:
metrics = pd.read_csv(TABLE_DIR / 'itemcf_metrics.csv')
display(metrics[['split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}'}))

,split,users,Precision@10,Recall@10,NDCG@10
0,validation,5776,6.00%,7.91%,0.0847
1,test,5765,5.68%,7.18%,0.0779


## 3. 按用户推荐电影

修改 `USER_ID` 后运行下面的单元格，显示用户尚未评分电影的标题和类型。
默认推荐 5 部，满足项目要求；演示使用测试期开始前的已知历史。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = recommend_for_user(USER_ID, stage='test', k=TOP_K)
display(recommendations[['rank', 'title', 'genres']])

,rank,title,genres
0,1,Raiders of the Lost Ark (1981),Action|Adventure
1,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,3,"Shawshank Redemption, The (1994)",Drama
3,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,5,Pulp Fiction (1994),Crime|Drama


算法参考：Sarwar et al. (2001), [Item-based Collaborative Filtering Recommendation Algorithms](https://ra.ethz.ch/cdstore/www10/papers/519/sdm2.html)。本项目采用正反馈二值输入的 Top-N 变体。